In [ ]:
%%html
<style>
    h1 {color:darkblue}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Course Outline
0. Part 0 — Intro/Setup
1. **Part 1 — OpenAI Python APIs**
   * 01-01. Text Generation Via the Responses API
   * 01-02. **Speech Recognition, Speech Synthesis, and Closed Captions**
   * 01-03. Images: Generation and Style Transfer
   * 01-04. Content Moderation
   * 01-05. Generating Code with a Codex Model and the Responses API
2. Part 2 — OpenAI Agents SDK
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Speech Recognition, Speech Synthesis, and Closed Captions
* Speech to text (STT) 
    * Transcribe speech in many spoken languages to text in those languages
    * Transcribe and translate non-English speech directly to English text
* Text to speech (TTS) — Synthesizing audio from text
    * Synthesizes speech in 57 languages, 13 natural-sounding voices
* Closed captions — Generating WebVTT captions from an audio track

---


## `import` Modules and Setup `OpenAI` Client Object

In [ ]:
import IPython
from openai import OpenAI 
from pathlib import Path
client = OpenAI() # create the OpenAI client object

<hr/>

## Speech-to-Text — English Audio Transcription
* Use speech recognition to transcribe audio file to text
    * Audio from my Python Fundamentals video course Lesson 1 test-drive of running a script
* Submit OpenAI Audio API transcription request via `OpenAI` object's `audio.transcriptions.create` method
* `gpt-4o-transcribe` model
    * Transcribe speech in ~100 languages
    * 2.46% English word error rate (WER) on FLEURS (Few-shot Learning Evaluation of Universal Representations of Speech) benchmark used to evaluate audio transcription models
    * Approximately half that of human transcribers
* https://developers.openai.com/api/docs/guides/speech-to-text

---
### `speech_to_text` Function
* [Open speech_to_text.py](./source/speech_to_text.py)
* Receives `OpenAI` client object and path to an audio file
* Calls `client.audio.transcriptions.create` with `gpt-4o-transcribe` model and audio file
* Returns transcribed text
* SDK encodes audio file's bytes for you, unlike with images and Responses API

In [ ]:
from source.speech_to_text import speech_to_text

In [ ]:
# audio from a Python Fundamentals "test-drive" video demoing script execution
audio_path = Path('resources') / 'Python2e_01_02.m4a'

In [ ]:
transcript = speech_to_text(client, audio_path)

In [ ]:
transcript

## Text-to-Speech
* Over 7000 natural languages worldwide
* CIA World Factbook: Top 11 languages account for 44.5% of the world's population
* OpenAI's Audio API can synthesize speech from text in 13 voices across 57 languages
  * `alloy`, `ash`, `ballad`, `cedar`, `coral`, `echo`, `fable`, `marin`, `nova`, `onyx`, `sage`, `shimmer`, `verse`
    * Voice availability depends on model
    * Now recommending `marin` or `cedar` voices
* Voices are optimized for English but also work for other languages
    * Experiment at https://openai.fm/
* https://developers.openai.com/api/docs/guides/text-to-speech

---
### `text_to_speech` Function
* [Open text_to_speech.py](./source/text_to_speech.py)
* Submit **OpenAI Audio API speech synthesis** requests via the `OpenAI` object's `audio.speech.create` method
* arguments:
  * `client` — the OpenAI object used to access the APIs
  * `text` — the string to synthesize into speech
  * `path` — a Path object containing the location of the file in which the synthesized audio should be stored
  * `voice` — the string name of the voice to use
  * `instructions` — an optional prompt containing instructions that help stylize the voice
      * Does not always work
* Returns synthesized speech as MP3 audio file by default
    * MP3 balances file size with audio quality 
    * Removes audio data less likely to be heard by humans
    * Achieves up to 90% reductions in file size
* `response` object's `write_to_file` method stores audio 



### Synthesizing and Playing English Speech in a Happy Tone

In [ ]:
from source.text_to_speech import text_to_speech

In [ ]:
english_text = (
    'Today was a beautiful day. Tomorrow looks like bad weather.')

In [ ]:
english_happy_path = (Path('resources') / 'outputs' / 
    'english_happy.mp3')

In [ ]:
text_to_speech(client, text=english_text, 
    path=english_happy_path, voice='marin', 
    instructions='Speak in a cheerful and positive tone.')

In [ ]:
IPython.display.Audio(filename=english_happy_path) # displays an audio player

### Synthesizing and Playing English Speech in an Evil Tone

In [ ]:
english_evil_path = (Path('resources') / 'outputs' / 
    'english_evil.mp3')

In [ ]:
text_to_speech(client, text=english_text, 
    path=english_evil_path, voice='ash',
    instructions="""Speak in the tone of an evil  
        supervillain and end with an evil laugh.""")

In [ ]:
IPython.display.Audio(filename=english_evil_path)

### Synthesizing and Playing Spanish Speech
* Reads text from the Spanish translation file created by `01-01-text-generation.ipynb`.

In [ ]:
spanish_text_path = Path('resources') / 'outputs' / 'spanish_text.txt'
spanish_text = spanish_text_path.read_text(encoding='utf-8')
print(spanish_text)
spanish_path = Path('resources') / 'outputs' / 'spanish.mp3'

In [ ]:
text_to_speech(client, text=spanish_text, 
    path=spanish_path, voice='cedar')

In [ ]:
IPython.display.Audio(filename=spanish_path)

### Synthesizing and Playing Japanese Speech
* Reads text from the Japanese translation file created by `01-01-text-generation.ipynb`.


In [ ]:
japanese_text_path = Path('resources') / 'outputs' / 'japanese_text.txt'
japanese_text = japanese_text_path.read_text(encoding='utf-8')
print(japanese_text)
japanese_path = Path('resources') / 'outputs' / 'japanese.mp3'

In [ ]:
text_to_speech(client, text=japanese_text, 
    path=japanese_path, voice='cedar')

In [ ]:
IPython.display.Audio(filename=japanese_path)

---
## Speech-to-Text Example — Closed Captions from a Video’s Audio Track
* Can’t upload video directly to OpenAI APIs to generate captions
* Can perform the following steps:
    * Use a tool like QuickTime on macOS or the cross-platform VLC Media Player (https://www.videolan.org/) to save video's audio track as an audio file
    * Use OpenAI Audio API's transcription with the whisper-1 model, asking for captions in WebVTT (Web Video Text Track) format
* Generates WebVTT closed captions from audio
    * Transcription is split into segments with timestamps
    * Save the WebVTT text into a .vtt file
    * Open the video in an application that supports closed captions and play it
    * VLC Media Player allows you to select a video and its .vtt file 
---
### `speech_to_vtt` Function 
* [Open speech_to_vtt.py](./source/speech_to_vtt.py)
* Receives `OpenAI` client object and a `Path` to a file containing audio track 
* Older `whisper-1` model has built-in support for transcribing audio in WebVTT format
* Pass `response_format='vtt'` to the client object's `audio.transcriptions.create` method

In [ ]:
from source.speech_to_vtt import speech_to_vtt

In [ ]:
caption_audio_path = Path('resources') / 'Python2e_01_02.m4a'

In [ ]:
captions = speech_to_vtt(client, caption_audio_path)

In [ ]:
vtt_path = Path('resources') / 'outputs' / 'Python2e_01_02.vtt'
vtt_path.parent.mkdir(parents=True, exist_ok=True)
vtt_path.write_text(captions, encoding='utf-8')
vtt_path

In [ ]:
print(captions)

#
---
&copy; 2026 by Deitel & Associates, Inc. All Rights Reserved. 